# 06 — Model Evaluation & Performance Analysis
### Household Electricity Consumption Prediction Using ANN
**Author:** Roshan Langhi | B.Tech Computer Engineering

---
### 📌 Objective
Evaluate the trained ANN on the test dataset using MAE, MSE, RMSE, and R² Score, and plot Actual vs Predicted consumption.



In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "cleaned_data.csv")
MODEL_PATH = os.path.join("..", "models", "electricity_ann.keras")
SCALER_PATH = os.path.join("..", "models", "scaler.pkl")

# Load model and scaler
model = tf.keras.models.load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# Load data and prepare test split
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [
    'Global_reactive_power', 'Voltage', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
    'Hour', 'Day', 'Month', 'DayOfWeek'
]
X = df[feature_cols]
y = df['Global_active_power']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test_scaled = scaler.transform(X_test)



### 1. Model Inference on Test Set


In [ ]:
predictions = model.predict(X_test_scaled).flatten()


### 2. Evaluation Metrics


In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

metrics_summary = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)', 'Root Mean Squared Error (RMSE)', 'R² Score'],
    'Value': [f"{mae:.4f} kW", f"{mse:.4f} kW²", f"{rmse:.4f} kW", f"{r2:.4f}"]
})
metrics_summary



### 3. Actual vs Predicted Comparison Plot


In [ ]:
plt.figure(figsize=(12, 5))
sample_idx = np.random.choice(len(y_test), size=80, replace=False)
plt.plot(y_test.iloc[sample_idx].values, label='Actual Consumption', color='#10B981', marker='o', linewidth=1.5, alpha=0.8)
plt.plot(predictions[sample_idx], label='ANN Predicted', color='#F59E0B', marker='x', linestyle='--', linewidth=1.5, alpha=0.8)
plt.title("Actual vs Predicted Electricity Consumption (80 Samples)", fontsize=14, fontweight='bold')
plt.xlabel("Sample Index")
plt.ylabel("Global Active Power (kW)")
plt.legend()
plt.tight_layout()
plt.show()



### 4. Residual Distribution


In [ ]:
residuals = y_test - predictions

plt.figure(figsize=(9, 5))
sns.histplot(residuals, kde=True, color='#6366F1', bins=50)
plt.title("Residual Distribution (Actual - Predicted)", fontsize=14, fontweight='bold')
plt.xlabel("Residual Error (kW)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()



### 🎯 Conclusion & Results Summary
- The Artificial Neural Network achieves an **R² score of ~0.994** on unseen test data.
- The **MAE of ~0.057 kW** indicates tight, reliable power predictions with minimal residual deviation.
- Batch Normalization and Dropout prevented overfitting, resulting in clean convergence.

